In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, classification_report
import pandas as pd
import numpy as np

In [ ]:


def process_dataset(input_path, output_path):
    # Read CSV (no header assumed)
    df = pd.read_csv(input_path, header=None)

    # Rename columns
    df.columns = ["label", "title", "review"]

    # Convert label: keep as-is (1 = negative, 2 = positive)
    # OR uncomment below if you want 0/1 labels instead
    # df["label"] = df["label"].map({1: 0, 2: 1})

    # Create id column
    df.insert(0, "id", range(len(df)))

    # Reorder columns explicitly
    df = df[["id", "title", "review", "label"]]

    # Save to CSV
    df.to_csv(output_path, index=False)

    return df


In [ ]:
train_processed = process_dataset("train.csv", "train_processed.csv")
test_processed  = process_dataset("test.csv", "test_processed.csv")

print("Train shape:", train_processed.shape)
print("Test shape:", test_processed.shape)

train_processed.head()
test_processed.head()

Train shape: (3600000, 4)
Test shape: (400000, 4)


,id,title,review,label
0,0,Great CD,My lovely Pat has one of the GREAT voices of h...,2
1,1,One of the best game music soundtracks - for a...,Despite the fact that I have only played a sma...,2
2,2,Batteries died within a year ...,I bought this charger in Jul 2003 and it worke...,1
3,3,"works fine, but Maha Energy is better",Check out Maha Energy's website. Their Powerex...,2
4,4,Great for the non-audiophile,Reviewed quite a bit of the combo players and ...,2


In [ ]:
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

def create_sentence_dataset_streaming(
    input_path,
    output_path,
    chunk_size=10_000
):
    first_chunk = True

    for chunk in pd.read_csv(input_path, chunksize=chunk_size):
        rows = []

        for _, row in chunk.iterrows():
            review_id = row["id"]
            label = row["label"]
            review = str(row["review"])

            for sent in sent_tokenize(review):
                sent = sent.strip()
                if sent:
                    rows.append((review_id, sent, label))

        sentence_df = pd.DataFrame(
            rows,
            columns=["id", "sentence", "label"]
        )

        sentence_df.to_csv(
            output_path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False
        )

        first_chunk = False
        del sentence_df, rows


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
create_sentence_dataset_streaming(
    "train_processed.csv",
    "train_sentences.csv",
    chunk_size=5000  # reduce if still tight
)

create_sentence_dataset_streaming(
    "test_processed.csv",
    "test_sentences.csv",
    chunk_size=5000  # reduce if still tight
)

In [3]:
train_df = pd.read_csv("train_sentences.csv")
print("Train sentences shape:", train_df.shape)

test_df = pd.read_csv("test_sentences.csv")
print("Test sentences shape:", test_df.shape)

print("Train - Min / max sentences per review")
print("Min sentences per review:",
      train_df.groupby("id").size().min())

print("Max sentences per review:",
      train_df.groupby("id").size().max())

print("Test - Min / max sentences per review")
print("Min sentences per review:",
      test_df.groupby("id").size().min())

print("Max sentences per review:",
      test_df.groupby("id").size().max())


Train sentences shape: (16808081, 3)
Test sentences shape: (1865666, 3)
Train - Min / max sentences per review
Min sentences per review: 1
Max sentences per review: 85
Test - Min / max sentences per review
Min sentences per review: 1
Max sentences per review: 40


In [4]:
def remove_single_sentence_docs(df):
    # Count number of sentences per document
    doc_counts = df.groupby("id").size()

    # Keep only document IDs with more than 1 sentence
    valid_ids = doc_counts[doc_counts > 4].index

    # Filter dataframe
    return df[df["id"].isin(valid_ids)].reset_index(drop=True)


train_df = remove_single_sentence_docs(train_df)
test_df  = remove_single_sentence_docs(test_df)


print("Train min sentences:", train_df.groupby("id").size().min())
print("Test min sentences:", test_df.groupby("id").size().min())



print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train min sentences: 5
Test min sentences: 5
Train shape: (11245946, 3)
Test shape: (1247342, 3)


In [5]:
def dataset_overview(df, name="DATASET"):
    print(f"\n===== {name} OVERVIEW =====")
    print("Total samples:", len(df))
    print("\nClass Distribution (Counts):")
    print(df["label"].value_counts())

    print("\nClass Distribution (Percentage):")
    print((df["label"].value_counts(normalize=True) * 100).round(2))


dataset_overview(train_df, "TRAIN")
dataset_overview(test_df, "TEST")


===== TRAIN OVERVIEW =====
Total samples: 11245946

Class Distribution (Counts):
label
1    6098693
2    5147253
Name: count, dtype: int64

Class Distribution (Percentage):
label
1    54.23
2    45.77
Name: proportion, dtype: float64

===== TEST OVERVIEW =====
Total samples: 1247342

Class Distribution (Counts):
label
1    675060
2    572282
Name: count, dtype: int64

Class Distribution (Percentage):
label
1    54.12
2    45.88
Name: proportion, dtype: float64


In [6]:
def document_label_distribution(df, name="DATASET"):
    doc_labels = df.groupby("id")["label"].first()
    print(f"\n===== {name} DOCUMENT LABEL DISTRIBUTION =====")
    print(doc_labels.value_counts())
    print("\nPercentages:")
    print((doc_labels.value_counts(normalize=True) * 100).round(2))


document_label_distribution(train_df, "TRAIN")
document_label_distribution(test_df, "TEST")


===== TRAIN DOCUMENT LABEL DISTRIBUTION =====
label
1    868691
2    752006
Name: count, dtype: int64

Percentages:
label
1    53.6
2    46.4
Name: proportion, dtype: float64

===== TEST DOCUMENT LABEL DISTRIBUTION =====
label
1    96262
2    83754
Name: count, dtype: int64

Percentages:
label
1    53.47
2    46.53
Name: proportion, dtype: float64


In [7]:
def sample_balanced_documents_sequential(
    df,
    id_col="id",
    label_col="label",
    n_docs_per_class=500,
):
    """
    Sequentially samples documents (not sentences) so that:
    - Documents are balanced across labels
    - All sentences for a document are kept
    - Order is preserved (no random sampling)
    """

    sampled_dfs = []

    for label in [1, 2]:
        # Get document ids in original order
        doc_ids = (
            df[df[label_col] == label]
            .groupby(id_col)
            .size()
            .index
        )

        # Take first N document IDs sequentially
        selected_doc_ids = doc_ids[:n_docs_per_class]

        # Keep ALL sentences for selected documents
        sampled_label_df = df[df[id_col].isin(selected_doc_ids)]
        sampled_dfs.append(sampled_label_df)

    balanced_df = pd.concat(sampled_dfs).reset_index(drop=True)

    return balanced_df

train_small = sample_balanced_documents_sequential(
    train_df,
    id_col="id",
    label_col="label",
    n_docs_per_class=4500
)

test_small = sample_balanced_documents_sequential(
    test_df,
    id_col="id",
    label_col="label",
    n_docs_per_class=4500
)

print("Train shape:", train_small.shape)
print("Test shape:", test_small.shape)

print("Train label counts:")
print(train_small.groupby("label")["id"].nunique())

print("Train min sentences per doc:",
      train_small.groupby("id").size().min())

print("Test min sentences per doc:",
      test_small.groupby("id").size().min())





Train shape: (62443, 3)
Test shape: (62780, 3)
Train label counts:
label
1    4500
2    4500
Name: id, dtype: int64
Train min sentences per doc: 5
Test min sentences per doc: 5


In [8]:
model = SentenceTransformer("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [9]:
def encode_sentences(df):
    embeddings = model.encode(
        df["sentence"].tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True
    )
    return embeddings

train_embeddings = encode_sentences(train_small)
test_embeddings  = encode_sentences(test_small)

Batches:   0%|          | 0/1952 [00:00<?, ?it/s]

Batches:   0%|          | 0/1962 [00:00<?, ?it/s]

In [10]:
svm = SVC(
    kernel="rbf",
    probability=True,
    class_weight="balanced",   # <-- IMPORTANT
    random_state=42
)
svm.fit(train_embeddings, train_small["label"])

SVC(class_weight='balanced', probability=True, random_state=42)

In [11]:
positive_label = 2
pos_idx = list(svm.classes_).index(positive_label)

# svm.classes_ tells you the exact order of classes used internally
# You explicitly select probability of label = 2
# Works even if class order changes
# No silent bugs

train_small["svm_prob"] = svm.predict_proba(train_embeddings)[:, pos_idx]
#train_small["svm_prob"] = svm.predict_proba(train_embeddings)[:, 1]
test_small["svm_prob"]  = svm.predict_proba(test_embeddings)[:, pos_idx]

In [12]:
import joblib

joblib.dump(svm, "svm_rbf_model_amazon_reviews.joblib")

['svm_rbf_model_amazon_reviews.joblib']

In [13]:
def build_doc_vectors(df, prob_col="svm_prob"):
    grouped = df.groupby("id")[prob_col].apply(list)
    min_len = grouped.apply(len).min()

    X = np.array([
        vals[:min_len] for vals in grouped
    ])
    return X, grouped.index.tolist(), min_len


X_train, train_ids, min_len = build_doc_vectors(train_small)
X_test, test_ids, _ = build_doc_vectors(test_small)

print(f"Truncated to {min_len} sentences per document")

Truncated to 5 sentences per document


In [14]:
def get_doc_labels(df, doc_ids): # Use integer IDs to match vectors
  return df.groupby("id")["label"].first().values


y_train = get_doc_labels(train_small, train_ids)
y_test  = get_doc_labels(test_small, test_ids)

In [15]:
doc_clf = LogisticRegression(class_weight="balanced",max_iter=1000)
doc_clf.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [16]:
def evaluate(model, X, y, name="SET"):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]

    print(f"\n===== {name} RESULTS =====")
    print("Accuracy:", accuracy_score(y, preds))
    print("F1 Score:", f1_score(y, preds))
    print("MCC:", matthews_corrcoef(y, preds))
    print("\nClassification Report:\n", classification_report(y, preds))


evaluate(doc_clf, X_train, y_train, "TRAIN")
evaluate(doc_clf, X_test, y_test, "TEST")


===== TRAIN RESULTS =====
Accuracy: 0.9145555555555556
F1 Score: 0.9148488539475141
MCC: 0.8291307852873634

Classification Report:
               precision    recall  f1-score   support

           1       0.91      0.92      0.91      4500
           2       0.92      0.91      0.91      4500

    accuracy                           0.91      9000
   macro avg       0.91      0.91      0.91      9000
weighted avg       0.91      0.91      0.91      9000


===== TEST RESULTS =====
Accuracy: 0.8626666666666667
F1 Score: 0.8620843561704976
MCC: 0.7253591959840041

Classification Report:
               precision    recall  f1-score   support

           1       0.87      0.86      0.86      4500
           2       0.86      0.87      0.86      4500

    accuracy                           0.86      9000
   macro avg       0.86      0.86      0.86      9000
weighted avg       0.86      0.86      0.86      9000

